# 01 — Brasil no Anthropic Economic Index

Análise exploratória do uso do Claude no Brasil comparado com outros países,
baseada nas 3 releases públicas do Anthropic Economic Index (Ago 2025 → Fev 2026).

**Dataset:** [Anthropic/EconomicIndex](https://huggingface.co/datasets/Anthropic/EconomicIndex)  
**Autor:** Gabriella Pinheiro  
**Data:** Abril 2026

---

## Estrutura do notebook

1. Setup
2. Análise 1 — Índice per capita do Brasil ao longo do tempo
3. Análise 2 — Brasil vs LATAM vs mundo
4. Análise 3 — O que os brasileiros mais pedem ao Claude
5. Análise 4 — Automação vs Augmentação
6. Análise 5 — Mapa mundial interativo (HTML)
7. Export dos outputs

---
## 1. Setup

In [ ]:
%pip install -q pandas pyarrow matplotlib seaborn plotly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Estilo dos gráficos
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'
BLUE   = '#1565C0'   # Brasil
GRAY   = '#BDBDBD'   # outros países
RED    = '#C62828'   # destaque negativo
GREEN  = '#2E7D32'   # destaque positivo

# Países comparáveis
LATAM = ['BRA', 'ARG', 'MEX', 'COL', 'CHL', 'PER', 'URY']
EMERGING = ['BRA', 'ARG', 'MEX', 'COL', 'CHL', 'IND', 'ZAF', 'TUR', 'IDN']
LABELS = {
    'BRA': 'Brasil',     'ARG': 'Argentina', 'MEX': 'México',
    'COL': 'Colômbia',   'CHL': 'Chile',     'PER': 'Peru',
    'URY': 'Uruguai',    'IND': 'Índia',     'ZAF': 'África do Sul',
    'TUR': 'Turquia',    'IDN': 'Indonésia', 'USA': 'EUA',
    'GBR': 'Reino Unido','DEU': 'Alemanha',  'ISR': 'Israel',
    'SGP': 'Singapura',  'AUS': 'Austrália', 'KOR': 'Coreia do Sul',
}

# Carrega dados
df_all = pd.read_parquet(DATA_DIR / 'aei_all_releases.parquet')
df_aug = pd.read_parquet(DATA_DIR / 'aei_aug2025.parquet')
df_feb = pd.read_parquet(DATA_DIR / 'aei_feb2026.parquet')

print(f'Dataset completo: {df_all.shape}')
print(f'Releases: {df_all["release_label"].unique()}')
print(f'Brasil presente: {"BRA" in df_all["geo_id"].values}')

---
## Análise 1 — Índice per capita do Brasil ao longo do tempo

O `usage_per_capita_index` mede quantas vezes acima (ou abaixo) da média global
o Brasil usa o Claude, ajustado pelo tamanho da população em idade ativa.
Valor > 1 = acima da média global. Valor < 1 = abaixo.

In [ ]:
# Série temporal: Brasil + países comparáveis
df_trend = df_all[
    (df_all['geo_id'].isin(LATAM + ['USA', 'IND', 'ZAF'])) &
    (df_all['geography'] == 'country') &
    (df_all['facet']     == 'country') &
    (df_all['variable']  == 'usage_per_capita_index')
].copy()

df_trend['label'] = df_trend['geo_id'].map(LABELS)
df_trend = df_trend.sort_values('release_date')

print('Índice per capita por país e release:')
pivot = df_trend.pivot_table(index='geo_id', columns='release_label', values='value')
pivot.index = pivot.index.map(LABELS)
pivot = pivot.sort_values('Fev 2026', ascending=False)
print(pivot.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for geo_id, group in df_trend.groupby('geo_id'):
    is_brazil = geo_id == 'BRA'
    color  = BLUE if is_brazil else GRAY
    lw     = 3.0  if is_brazil else 1.0
    alpha  = 1.0  if is_brazil else 0.6
    zorder = 5    if is_brazil else 2
    label  = LABELS.get(geo_id, geo_id)

    ax.plot(
        group['release_label'], group['value'],
        marker='o', color=color, linewidth=lw,
        alpha=alpha, zorder=zorder,
        markersize=8 if is_brazil else 4,
        label=label
    )

    # Anota apenas o Brasil
    if is_brazil:
        for _, row in group.iterrows():
            ax.annotate(
                f"{row['value']:.2f}×",
                xy=(row['release_label'], row['value']),
                xytext=(0, 14), textcoords='offset points',
                ha='center', fontsize=10,
                color=BLUE, fontweight='bold'
            )

# Linha de média global
ax.axhline(1, color='#78909C', linestyle='--', linewidth=1.2,
           label='Média global (= 1)', zorder=1)

ax.set_ylabel('Índice de uso per capita', fontsize=11)
ax.set_title(
    'Adoção do Claude — índice per capita ao longo do tempo\n'
    'Brasil vs países comparáveis (Ago 2025 → Fev 2026)',
    fontsize=13, pad=12
)
ax.legend(loc='upper left', fontsize=8, ncol=3, framealpha=0.7)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_brazil_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: outputs/01_brazil_trend.png')

---
## Análise 2 — Brasil vs LATAM: ranking regional

Como o Brasil se posiciona dentro da América Latina em termos de adoção per capita?

In [ ]:
# Ranking LATAM na release mais recente (Fev 2026)
df_latam = df_all[
    (df_all['geo_id'].isin(LATAM)) &
    (df_all['geography']     == 'country') &
    (df_all['facet']         == 'country') &
    (df_all['variable']      == 'usage_per_capita_index') &
    (df_all['release_label'] == 'Fev 2026')
].copy()

df_latam['label'] = df_latam['geo_id'].map(LABELS)
df_latam = df_latam.sort_values('value', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))

colors = [BLUE if g == 'BRA' else GRAY for g in df_latam['geo_id']]
bars = ax.barh(df_latam['label'], df_latam['value'], color=colors, height=0.6)

# Valor ao lado de cada barra
for bar, val in zip(bars, df_latam['value']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}×', va='center', fontsize=10)

ax.axvline(1, color='#78909C', linestyle='--', linewidth=1.2, label='Média global')
ax.set_xlabel('Índice de uso per capita (média global = 1)', fontsize=10)
ax.set_title('Adoção do Claude na América Latina — Fev 2026', fontsize=13)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlim(0, df_latam['value'].max() * 1.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_latam_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: outputs/02_latam_ranking.png')

---
## Análise 3 — O que os brasileiros mais pedem ao Claude

Top 10 categorias de request no Brasil vs média global (Ago 2025 — única release com request_pct_index).

In [ ]:
# Top 10 requests do Brasil (Ago 2025)
df_req_bra = df_aug[
    (df_aug['geo_id']    == 'BRA') &
    (df_aug['facet']     == 'request') &
    (df_aug['variable']  == 'request_pct') &
    (df_aug['level']     == 0)
].nlargest(10, 'value')[['cluster_name', 'value']].copy()

# Top 10 requests global (Ago 2025)
df_req_glob = df_aug[
    (df_aug['geography'] == 'global') &
    (df_aug['facet']     == 'request') &
    (df_aug['variable']  == 'request_pct') &
    (df_aug['level']     == 0)
].nlargest(10, 'value')[['cluster_name', 'value']].copy()

print('Top 10 requests — Brasil:')
for i, row in df_req_bra.iterrows():
    print(f'  {row["cluster_name"][:70]:70s}  {row["value"]*100:.1f}%')

print('\nTop 10 requests — Global:')
for i, row in df_req_glob.iterrows():
    print(f'  {row["cluster_name"][:70]:70s}  {row["value"]*100:.1f}%')

In [ ]:
# Gráfico comparativo Brasil vs Global
# Une os dois top-10 em um único conjunto de categorias
top_cats = pd.concat([
    df_req_bra.assign(group='Brasil'),
    df_req_glob.assign(group='Global')
])

# Abreviações para o gráfico
def short(name, max_len=45):
    return name[:max_len] + '...' if len(name) > max_len else name

top_cats['label'] = top_cats['cluster_name'].apply(short)

# Pivota
pivot_req = (
    top_cats.pivot_table(index='label', columns='group', values='value')
    .fillna(0)
    .sort_values('Brasil', ascending=True)
)

fig, ax = plt.subplots(figsize=(11, 6))
x = range(len(pivot_req))
w = 0.38

ax.barh([i - w/2 for i in x], pivot_req['Brasil'] * 100,
        height=w, label='Brasil', color=BLUE, alpha=0.85)
ax.barh([i + w/2 for i in x], pivot_req.get('Global', 0) * 100,
        height=w, label='Global', color=GRAY, alpha=0.85)

ax.set_yticks(list(x))
ax.set_yticklabels(pivot_req.index, fontsize=9)
ax.set_xlabel('% do uso total', fontsize=10)
ax.set_title(
    'O que as pessoas pedem ao Claude — Brasil vs Global\n'
    'Top categorias de request (Ago 2025)',
    fontsize=13
)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_requests_bra_vs_global.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: outputs/03_requests_bra_vs_global.png')

---
## Análise 4 — Automação vs Augmentação

A Anthropic classifica o uso em dois grandes padrões:
- **Automação**: o modelo executa a tarefa de forma autônoma (directive + feedback_loop)
- **Augmentação**: o humano mantém controle, IA apoia (validation + task_iteration + learning)

Disponível apenas em Ago 2025.

In [ ]:
# Automação vs Augmentação por país (Ago 2025)
df_auto = df_aug[
    (df_aug['geography'] == 'country') &
    (df_aug['facet']     == 'collaboration_automation_augmentation') &
    (df_aug['variable'].isin(['automation_pct', 'augmentation_pct'])) &
    (df_aug['geo_id'].isin(LATAM + ['USA', 'IND', 'ZAF', 'GBR']))
].copy()

df_auto['label'] = df_auto['geo_id'].map(LABELS)
df_auto['variable'] = df_auto['variable'].str.replace('_pct', '')

pivot_auto = (
    df_auto.pivot_table(index=['geo_id', 'label'], columns='variable', values='value')
    .reset_index()
    .sort_values('automation', ascending=False)
)

print('Automação vs Augmentação por país (Ago 2025):')
print(pivot_auto[['label', 'automation', 'augmentation']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(pivot_auto))
w = 0.38

bar1 = ax.bar([i - w/2 for i in x], pivot_auto['automation'],
              width=w, label='Automação', color=RED, alpha=0.8)
bar2 = ax.bar([i + w/2 for i in x], pivot_auto['augmentation'],
              width=w, label='Augmentação', color=BLUE, alpha=0.8)

# Destaca Brasil
bra_pos = pivot_auto[pivot_auto['geo_id'] == 'BRA'].index
if len(bra_pos) > 0:
    pos = list(pivot_auto['geo_id']).index('BRA')
    ax.axvspan(pos - 0.5, pos + 0.5, alpha=0.08, color=BLUE, zorder=0)

ax.set_xticks(list(x))
ax.set_xticklabels(pivot_auto['label'], rotation=25, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
ax.set_title(
    'Automação vs Augmentação no uso do Claude — Ago 2025\n'
    'Brasil em destaque',
    fontsize=13
)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_automation_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: outputs/04_automation_augmentation.png')

---
## Análise 5 — Mapa mundial interativo

Heatmap do `usage_per_capita_index` por país na release mais recente (Fev 2026).
Exportado como HTML interativo — hover mostra país e valor.

In [ ]:
# Dados do mapa — release Fev 2026
df_map = df_all[
    (df_all['geography']     == 'country') &
    (df_all['facet']         == 'country') &
    (df_all['variable']      == 'usage_per_capita_index') &
    (df_all['release_label'] == 'Fev 2026')
][['geo_id', 'country_name', 'value']].dropna(subset=['value']).copy()

df_map['hover'] = df_map.apply(
    lambda r: f"{r['country_name'] or r['geo_id']}<br>Índice: {r['value']:.2f}×", axis=1
)

print(f'Países no mapa: {len(df_map)}')
print(f'Brasil: {df_map[df_map["geo_id"]=="BRA"]["value"].values}')
print(f'Top 5:')
print(df_map.nlargest(5, 'value')[['geo_id', 'country_name', 'value']].to_string(index=False))

In [ ]:
# Mapa choropleth interativo com Plotly
fig_map = px.choropleth(
    df_map,
    locations='geo_id',
    color='value',
    hover_name='country_name',
    hover_data={'geo_id': False, 'value': ':.2f'},
    color_continuous_scale=[
        [0.0,  '#E3F2FD'],
        [0.3,  '#90CAF9'],
        [0.6,  '#1565C0'],
        [1.0,  '#0D47A1'],
    ],
    range_color=[0, df_map['value'].quantile(0.95)],
    labels={'value': 'Índice per capita'},
    title='Uso do Claude por País — Índice per capita · Fev 2026<br><sup>Média global = 1× · Fonte: Anthropic Economic Index</sup>',
)

# Destaca Brasil com borda
bra_val = df_map[df_map['geo_id'] == 'BRA']['value'].values
if len(bra_val) > 0:
    fig_map.add_trace(go.Choropleth(
        locations=['BRA'],
        z=[bra_val[0]],
        colorscale=[[0, '#1565C0'], [1, '#1565C0']],
        showscale=False,
        marker_line_color='#FF6F00',
        marker_line_width=2,
        hovertemplate='Brasil<br>Índice: %{z:.2f}×<extra></extra>'
    ))

fig_map.update_layout(
    geo=dict(showframe=False, showcoastlines=True, coastlinecolor='white'),
    coloraxis_colorbar=dict(title='Índice<br>per capita'),
    margin=dict(l=0, r=0, t=60, b=0),
    height=500,
)

# Salva HTML
map_path = OUTPUT_DIR / '05_world_heatmap.html'
fig_map.write_html(str(map_path))
print(f'Mapa salvo: {map_path}')

fig_map.show()

---
## 7. Resumo dos outputs

Todos os arquivos salvos em `outputs/`:

In [ ]:
outputs = list(OUTPUT_DIR.glob('0*'))
print(f'{len(outputs)} arquivo(s) gerado(s) em {OUTPUT_DIR}:\n')
for f in sorted(outputs):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s}  {size_kb:8.1f} KB')

print()
print('Para o post do LinkedIn:')
print('  01_brazil_trend.png        → gráfico de linha temporal')
print('  02_latam_ranking.png       → ranking LATAM')
print('  03_requests_bra_vs_global.png → o que brasileiros pedem')
print('  04_automation_augmentation.png → automação vs augmentação')
print('  05_world_heatmap.html      → mapa interativo (abre no browser)')

---
## Insights para o post

Com base nas análises acima, os principais achados são:

1. **O Brasil usa menos Claude do que o esperado pelo tamanho da população** — índice abaixo de 1× nas 3 releases
2. **O índice caiu entre Ago 2025 e Fev 2026** — de ~0.93× para ~0.79× (dado oficial da Anthropic)
3. **No LATAM, o Brasil não lidera** — Chile e Uruguai têm índices per capita maiores
4. **O uso no Brasil é dominado por tarefas jurídicas e acadêmicas** — diferente do padrão global (coding + software)
5. **63.9% do uso no Brasil é automação** — o modelo executa sem supervisão humana contínua
   — acima da média, o que pode indicar uso mais instrumental do que colaborativo

A pergunta que fica: o Brasil está atrasado na adoção ou simplesmente tem um perfil de uso diferente?